## Classified Project: "BIRD IS THE WORD"
**(or just birdstrikes - a look at the data - course: Data Analysis & Visualizaion)**

**Initial thoughts (some lose asumptions)**

In answering the first question of when during the 24h cycle and what phase of flight strikes occur. I'm taking the following into account:

'Phase of flight'

The 'Time of day' column contains 4 unique values.

Time of year can be derived from the "Flight Date" column. Here I would like to examine if bird migration has any impact on birdstrikes. Which brings us to the next column in the DataFrame.**

Locations can be derived from the'Origin State' column.

In [ ]:
# Importing the relevand libraries.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Applying Seaborn's darkgrid theme.
sns.set_theme(style='darkgrid')

# Creating a python list containing column names with dates for parsing.
strikes_raw = pd.read_csv(
    'data/birdstrikes.csv',
    parse_dates=['Flight Date']
)

# Loading the CSV and parsing the date_columns for the proper datatype in the DataFrame.
regions_raw = pd.read_csv(
    'data/us_regions.csv'
)

strikes = strikes_raw.copy()
regions = regions_raw.copy()



In [ ]:
# Displaying the number of rows and columns in the DataFrame.
print('Bird strikes:', strikes_raw.shape)

In [ ]:
strikes_raw.dtypes

In [ ]:
strikes_raw.head()

In [ ]:
# Inspecting the parsed column.
print(strikes['Flight Date'].dtype)

In [ ]:
before_parse_date = strikes_raw['Flight Date'].isna().sum()
missing_after = strikes['Flight Date'].isna().sum()

# There are apparently no NaT values (missing dates) during conversion (after parsing the Flight Date column).
print('Missing before:', before_parse_date)
print('Missing after:', missing_after)
print('New missing dates:', missing_after - before_parse_date)

In [ ]:
print('Earliest date:', strikes['Flight Date'].min())
print('Latest date:', strikes['Flight Date'].max())
# strikes['Flight Date'].describe()

In [ ]:
# Checking for duplicate rows
duplicate_after_first = strikes.duplicated().sum()

duplicate_rows = strikes[strikes.duplicated(keep=False)]

print('Duplicate rows after the first occurrence:', duplicate_after_first)
print('All rows involved in duplicate groups:', len(duplicate_rows))
print('Distinct records that are repeated:', len(duplicate_rows.drop_duplicates()))

**Anwering the first ("1.") requirement in the project PDF ("Investigate data"):**

The table consists of 10K rows and 14 columns.

Unit of observation (or grain of the data): In this table, one row is one reported collision between an aircraft and a bird (or "wildlife" since they are also not all birds).

I will mainly use `Flight Date`, `Phase of flight` and `Time of day` to examine 
when reported strikes occurred.

I will use `Effect Amount of damage`, `Cost Other`, `Cost Repair` and `Cost Total $` to investigate recorded damage and financial cost.

`Wildlife Size` and `Wildlife Species` will help compare 
the animals involved, while `Speed IAS in knots` may help describe the flight 
conditions.

I will use `Origin State` to merge the main table with 
`us_regions.csv`, creating a `Region` column for regional comparisons. 

`Airport Name` will provide location context and may be used for selected 
airport-level summaries.

In [ ]:
# Cleaning duplicate rows
strikes = (
    strikes
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_before_duplicates = len(strikes_raw) # Checking number of rows by querying the untouched dataframe.
print('Number of rows before duplicate rows removal:', rows_before_duplicates)

print('Number of rows after duplicate rows removal:', len(strikes))

**On the step of removing duplicate rows**
I found 24 exact duplicate rows after the first occurrence. Because all 14 fields matched, I treated the duplicates as redundant records and removed them, reducing the table from 10,000 to 9,976 rows. The table has no unique incident ID, so I cannot prove that the matching rows represent the same collision. They could represent separate incidents with identical data.

___

Moving on to the next steps of cleaning our strikes dataframe following duplicate rows removal.

In [ ]:
# Checking missing values for each column (after duplicate rows removal)
missing_summary = pd.DataFrame({
    'missing_count': strikes.isna().sum(),
    'missing_percent': (
        strikes.isna().mean() * 100
    ).round(1)
}).sort_values(
    'missing_percent',
    ascending=False
)

missing_summary

In [ ]:
# Missing does not prove that no damage occurred. Avoid treating this field as a complete measurement of damage severity.
strikes['Effect Amount of damage'].value_counts(dropna=False)

In [ ]:
# Making a comparison between missing damage descriptions and cost.
damage_missing = (strikes['Effect Amount of damage'].isna())

damage_cost_check = (
    strikes
    .groupby(damage_missing)['Cost Total $']
    .agg(['count', 'median', 'mean', 'max'])
    .round(2)
)

damage_cost_check.index = ['Damage description recorded', 'Damage description missing']

damage_cost_check

In [ ]:
# Missing description does not necessarily mean zero economic effect.
missing_damage_positive_cost = (
    damage_missing
    & (strikes['Cost Total $'] > 0)
)

print(
    'Missing damage description but positive total cost:',
    missing_damage_positive_cost.sum()
)

In [ ]:
# Adding a column to the working DataFrame (strikes).
strikes['Damage Description Recorded'] = (
    strikes['Effect Amount of damage'].notna()
)
print("Before adding 'Damage Description Recorded':", strikes_raw.shape)
print('After adding the column:', strikes.shape)

**A comment on the cleaning step above**
The "damage-description" field is missing for a majority of the strike reports. Rather than "inventing information", I left these values missing (for instance, I did not populate these values with “No damage”). Some reports with a missing damage description still have a positive recorded cost, showing that absence of a category or description is not equivalent to absence of damage.

___

**Next step in cleaning the data:** Categories C (14 of them) and B (1 entry) do not match the readable categories Minor, Medium, and Substantial. This is cause for further examination of the data.

In [ ]:
unclear_damage_codes = strikes[strikes['Effect Amount of damage'].isin(['B', 'C'])]

display(
    unclear_damage_codes[
        [
            'Flight Date',
            'Airport Name',
            'Effect Amount of damage',
            'Cost Total $'
        ]
    ]
)

print('Rows with unexplained B or C codes:', len(unclear_damage_codes))

In [ ]:
# Adding column 'Damage Category' for analysis. "Grouping" B and C under "Unclear damage code". Prevents presenting unexplained codes as if their meaning were known (when they are not known to us at present).
strikes['Damage Category'] = (
    strikes['Effect Amount of damage']
    .replace({
        'B': 'Unclear code',
        'C': 'Unclear code'
    })
    .fillna('Not recorded')
)

strikes['Damage Category'].value_counts()

Fifteen reports use the unexplained values B or C instead of descriptive readable damage categories. I preserved the original field as 'Damage Category' and created a separate analysis category called Unclear code. I did not translate the letters because I could not verify their meaning for this dataset version.
___

In [ ]:
speed_missing_count = (
    strikes['Speed IAS in knots'].isna().sum()
)

speed_missing_percent = (
    strikes['Speed IAS in knots']
    .isna()
    .mean()
    * 100
)

print('Missing speeds:', speed_missing_count)
print(
    'Missing percentage:',
    round(speed_missing_percent, 1)
)

In [ ]:
speed_missing_by_phase = pd.crosstab(
    strikes['Phase of flight'],
    strikes['Speed IAS in knots'].isna(),
    normalize='index'
) * 100

speed_missing_by_phase.columns = [
    'Speed recorded %',
    'Speed missing %'
]

speed_missing_by_phase.round(1)

In [ ]:
# An indicator (column) telling us if any speed entries are available (over rows).
strikes['Speed Recorded'] = (
    strikes['Speed IAS in knots'].notna()
)

# # Filtering recorded values for analysis.
# speed_known = strikes[
#     strikes['Speed IAS in knots'].notna()
# ].copy()

Airspeed is missing in a substantial share of reports. I retained the missing values because zero would mean a stationary aircraft, while mean or median filling would create fictional speeds and reduce the observed spread. Any analyses involving speed therefore use only reports with a recorded value and state the number of observations used.


In [ ]:
zero_speed = (
    strikes['Speed IAS in knots'] == 0
)

print('Rows with zero recorded speed, all', zero_speed.sum(), "of them. :)")

display(
    strikes.loc[
        zero_speed,
        [
            'Flight Date',
            'Airport Name',
            'Phase of flight',
            'Speed IAS in knots'
        ]
    ]
)

In [ ]:
# Creating a column, indicating questionable combinations between speed and "phase of flight".
strikes['Questionable Zero Speed'] = (
    (strikes['Speed IAS in knots'] == 0)
    & ~strikes['Phase of flight'].isin(
        ['Parked', 'Taxi', 'Landing Roll']
    )
)

In [ ]:
# Actually showing what we are talking about in regard to questionable combinations of speed and phase. It can also be mentioned that this is not an official aviation rule but merely for analysis.
print(
    'Questionable zero-speed rows:',
    strikes['Questionable Zero Speed'].sum()
)

display(
    strikes.loc[
        strikes['Questionable Zero Speed'],
        [
            'Flight Date',
            'Airport Name',
            'Phase of flight',
            'Speed IAS in knots'
        ]
    ]
)

In [ ]:
# Separate observation: Index 19 (row 20) shows NaN and not zero (or 0).
strikes['Speed IAS in knots'].notna().head(20)

In [ ]:
# Costs show consistency across the rows.
cost_mismatch = (
    strikes['Cost Total $']
    != (
        strikes['Cost Other']
        + strikes['Cost Repair']
    )
)

print(
    'Rows where total differs from its components:',
    cost_mismatch.sum()
)

In [ ]:
zero_cost = strikes['Cost Total $'] == 0
positive_cost = strikes['Cost Total $'] > 0

print('Zero recorded total cost:', zero_cost.sum())
print('Positive recorded total cost:', positive_cost.sum())

print(
    'Zero cost percentage:',
    round(zero_cost.mean() * 100, 1)
)

The dataset page identifies all three cost columns as numeric fields containing zero values, but provides no field-level explanation of whether zero means confirmed no cost or no cost supplied. Another analysis of FAA data explicitly treats missing cost as zero in sums and says it does not impute missing cost, illustrating why cost interpretation needs methodological caution.

In [ ]:
# Compairing 'Damage Category' with cost.
cost_damage_table = pd.crosstab(
    strikes['Damage Category'],
    zero_cost,
    margins=True
)

cost_damage_table.columns = [
    'Positive cost',
    'Zero cost',
    'Total'
]

cost_damage_table

It can be noted that some reports marked Minor, Medium, or Substantial still have a zero recorded cost. Therefore, zero cannot safely be interpreted as "nothing happened". I'll therefore leave the original cost values unchanged.

In [ ]:
strikes['Positive Recorded Cost'] = (
    strikes['Cost Total $'] > 0
)

In [ ]:
print(
    'Median across all reports:',
    strikes['Cost Total $'].median()
)

print(
    'Median among positive recorded costs:',
    strikes.loc[
        strikes['Cost Total $'] > 0,
        'Cost Total $'
    ].median()
)

Most reports contain a total cost of zero, but zero may represent either a genuine zero cost or the absence of a supplied cost estimate. I retained the original values and created a flag for positive recorded cost. I will therefore describe zero as “zero recorded cost,” not as proof that the incident caused no financial loss.

In [ ]:
category_columns = [
    'Phase of flight',
    'Time of day',
    'Wildlife Size',
    'Origin State',
    'Effect Amount of damage'
]

for column in category_columns:
    print(f'\n--- {column} ---')
    print(
        strikes[column]
        .value_counts(dropna=False)
    )

In [ ]:
# Checking if there is any hidden whitespace to be cleaned out (seemingly none).
for column in category_columns:
    non_missing = strikes[column].dropna()

    whitespace_count = (
        non_missing
        != non_missing.str.strip()
    ).sum()

    print(
        column,
        '- values with surrounding whitespace:',
        whitespace_count
    )

I inspected the main categorical columns for inconsistent capitalization, spelling and surrounding whitespace. The phase-of-flight, time-of-day, wildlife-size and state labels were internally consistent. The damage field was the exception because it contained missing values and 15 unexplained (B and C) letter codes.
No values need correction in four of the columns.

In [ ]:
print('Original shape:', strikes_raw.shape)
print('Cleaned shape:', strikes.shape)

print(
    'Remaining exact duplicates:',
    strikes.duplicated().sum()
)

print('\nData types:')
print(strikes.dtypes)

print('\nMissing values:')
print(
    strikes.isna()
    .sum()
    .sort_values(ascending=False)
)

print('\nCreated columns:')
created_columns = [
    'Damage Description Recorded',
    'Damage Category',
    'Speed Recorded',
    'Questionable Zero Speed',
    'Positive Recorded Cost'
]

print(created_columns)

In [ ]:
# Varifying that the original columns are still present in the working DataFrame.
print(
    'Original damage field still present:',
    'Effect Amount of damage' in strikes.columns
)

print(
    'Original cost field still present:',
    'Cost Total $' in strikes.columns
)

One final reservation remains: because the dataset has no unique incident ID, even completely identical rows cannot be proven to be the same real-world collision. Removing them is defensible, but it should be described as a documented assumption rather than absolute certainty.

## For now, that's it for cleaning this table of birdstrikes. Moving on to "next steps"...

___
___